# Análise Exploratória — Combustíveis ANP 2024

**Projeto Aplicado I — Mackenzie**  
Escopo: gasolina comum, etanol hidratado e diesel S10. Fonte: arquivos semestrais de preços de revenda da ANP.

Notebook descritivo que prepara os resultados para o storytelling da Etapa 3. Não identifica causas.

## 1. Bibliotecas e caminhos
Os dados brutos estão nos ZIPs versionados em `A2.py/dados`; não é necessário descompactá-los.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import pandas as pd
import matplotlib.pyplot as plt

pasta_atual = Path.cwd()
if (pasta_atual / "dados" / "ca-2024-01.zip").exists():
    projeto = pasta_atual
elif (pasta_atual / "A2.py" / "dados").exists():
    projeto = pasta_atual / "A2.py"
else:
    raise FileNotFoundError("Abra o notebook na pasta A2.py ou na raiz do repositório.")

dados_dir = projeto / "dados"
relatorios = projeto / "relatorio"
graficos = projeto / "graficos"
relatorios.mkdir(parents=True, exist_ok=True)
graficos.mkdir(parents=True, exist_ok=True)
entradas = [dados_dir / "ca-2024-01.zip", dados_dir / "ca-2024-02.zip"]
for arquivo in entradas:
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")
print("Arquivos:", *(a.name for a in entradas), sep="\n- ")

## 2. Leitura, limpeza e recorte
Unimos os dois semestres, convertemos preços e datas, selecionamos os combustíveis e mantemos os registros válidos de 2024.

In [ ]:
def ler_csv_anp(caminho):
    """Lê o CSV semestral guardado dentro do ZIP da ANP."""
    with ZipFile(caminho) as z:
        nomes = [n for n in z.namelist() if n.lower().endswith(".csv")]
        if not nomes:
            raise FileNotFoundError(f"Nenhum CSV em {caminho.name}")
        with z.open(nomes[0]) as f:
            return pd.read_csv(BytesIO(f.read()), sep=";", encoding="latin1",
                               dtype=str, low_memory=False)

dados = pd.concat([ler_csv_anp(a) for a in entradas], ignore_index=True)
dados.columns = (dados.columns.str.replace("ï»¿", "", regex=False)
                 .str.replace("\ufeff", "", regex=False).str.strip())
for c in ["Produto", "Municipio", "Estado - Sigla", "Regiao - Sigla"]:
    dados[c] = dados[c].astype("string").str.strip()
dados["Produto"] = dados["Produto"].str.upper()

preco = dados["Valor de Venda"].astype("string").str.replace("R$", "", regex=False).str.replace(" ", "", regex=False)
virgula = preco.str.contains(",", na=False)
preco.loc[virgula] = preco.loc[virgula].str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
dados["Preco"] = pd.to_numeric(preco, errors="coerce")
dados["Data da Coleta"] = pd.to_datetime(dados["Data da Coleta"], format="%d/%m/%Y", errors="coerce")

etapas = [{"Etapa": "Base original", "Registros": len(dados)}]
dados = dados[dados["Produto"].isin(["GASOLINA", "ETANOL", "DIESEL S10"])].copy()
etapas.append({"Etapa": "Após filtro dos combustíveis", "Registros": len(dados)})
dados = dados.dropna(subset=["Preco", "Data da Coleta"])
etapas.append({"Etapa": "Após remover preço/data inválidos", "Registros": len(dados)})
dados = dados[dados["Preco"] > 0].copy()
etapas.append({"Etapa": "Após manter preços positivos", "Registros": len(dados)})
dados = dados[dados["Data da Coleta"].dt.year == 2024].copy()
etapas.append({"Etapa": "Após filtro de 2024", "Registros": len(dados)})
if dados.empty:
    raise ValueError("Nenhum registro válido. Confira os arquivos e produtos.")
dados["Mes"] = dados["Data da Coleta"].dt.month
display(pd.DataFrame(etapas))
print("Período:", dados["Data da Coleta"].min(), "a", dados["Data da Coleta"].max())

## 3. Composição, estatísticas e possíveis extremos
O critério 1,5 × IQR sinaliza observações para inspeção, mas não as remove: valores extremos podem ser reais.

In [ ]:
display(dados["Produto"].value_counts().rename_axis("Produto").to_frame("Registros"))
colunas = ["Produto", "Preco", "Data da Coleta", "Regiao - Sigla", "Estado - Sigla", "Municipio"]
display(dados[colunas].isna().sum().rename("Ausências").to_frame())
estatisticas = dados.groupby("Produto")["Preco"].agg(
    Registros="count", Media="mean", Mediana="median", Minimo="min",
    Maximo="max", Desvio_Padrao="std"
).round(2)
display(estatisticas)

def contar_extremos(grupo):
    q1, q3 = grupo["Preco"].quantile([0.25, 0.75])
    iqr = q3 - q1
    fora = (grupo["Preco"] < q1 - 1.5 * iqr) | (grupo["Preco"] > q3 + 1.5 * iqr)
    return pd.Series({"Q1": q1, "Q3": q3, "Possíveis extremos": int(fora.sum())})

linhas_extremos = []
for produto, grupo in dados.groupby("Produto"):
    q1, q3 = grupo["Preco"].quantile([0.25, 0.75])
    iqr = q3 - q1
    fora = (grupo["Preco"] < q1 - 1.5 * iqr) | (grupo["Preco"] > q3 + 1.5 * iqr)
    linhas_extremos.append({"Produto": produto, "Q1": q1, "Q3": q3,
                            "Possíveis extremos": int(fora.sum())})
extremos = pd.DataFrame(linhas_extremos).set_index("Produto").round(2)
display(extremos)

## 4. Variação mensal
Comparamos as médias dos registros de janeiro e dezembro e exibimos a trajetória mensal. Não são médias ponderadas por volume vendido.

In [ ]:
mensal = dados.groupby(["Mes", "Produto"], as_index=False)["Preco"].mean().rename(columns={"Preco": "Preco medio"})
inicio_fim = mensal[mensal["Mes"].isin([1, 12])].pivot(index="Produto", columns="Mes", values="Preco medio")
inicio_fim.columns = ["Janeiro", "Dezembro"]
inicio_fim["Variacao percentual"] = (inicio_fim["Dezembro"] / inicio_fim["Janeiro"] - 1) * 100
display(inicio_fim.round(2))

ax = mensal.pivot(index="Mes", columns="Produto", values="Preco medio").plot(marker="o", figsize=(10, 5))
ax.set(title="Preço médio registrado por mês — Brasil, 2024", xlabel="Mês", ylabel="R$/litro")
ax.set_xticks(range(1, 13)); ax.grid(True, alpha=0.3); ax.legend(title="Combustível")
plt.tight_layout()
plt.savefig(graficos / "evolucao_mensal.png", dpi=160, bbox_inches="tight")
plt.show()

## 5. Diferenças regionais
As médias são calculadas sobre os registros disponíveis; a quantidade de observações pode diferir por produto e região.

In [ ]:
regional = dados.groupby(["Regiao - Sigla", "Produto"], as_index=False).agg(
    Preco_medio=("Preco", "mean"), Registros=("Preco", "size")
)
display(regional.pivot(index="Regiao - Sigla", columns="Produto", values="Preco_medio").round(2))
nomes = {"N": "Norte", "NE": "Nordeste", "CO": "Centro-Oeste", "SE": "Sudeste", "S": "Sul"}
tab = regional.pivot(index="Regiao - Sigla", columns="Produto", values="Preco_medio").rename(index=nomes)
ax = tab.plot(kind="bar", figsize=(10, 5))
ax.set(title="Preço médio registrado por região e combustível — 2024", xlabel="Região", ylabel="R$/litro")
ax.tick_params(axis="x", rotation=0); ax.legend(title="Combustível"); ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(graficos / "preco_medio_regiao.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Distribuição
O boxplot complementa as médias mostrando dispersão e pontos potencialmente extremos. Eles não são excluídos automaticamente.

In [ ]:
dados.boxplot(column="Preco", by="Produto", grid=False, figsize=(9, 5))
plt.suptitle(""); plt.title("Distribuição dos preços registrados — 2024")
plt.xlabel("Combustível"); plt.ylabel("R$/litro"); plt.tight_layout()
plt.savefig(graficos / "distribuicao_precos.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Exportação e síntese
A leitura principal é descritiva: as médias mensais da gasolina e do etanol variaram mais que a do diesel S10, e o Norte teve as maiores médias regionais dos três produtos. Isso não demonstra causas.

In [ ]:
dados.to_csv(relatorios / "dados_tratados.csv", index=False, encoding="utf-8-sig")
estatisticas.to_csv(relatorios / "estatisticas_combustiveis.csv", encoding="utf-8-sig")
mensal.to_csv(relatorios / "preco_medio_mensal.csv", index=False, encoding="utf-8-sig")
regional.to_csv(relatorios / "preco_medio_regiao.csv", index=False, encoding="utf-8-sig")
extremos.to_csv(relatorios / "diagnostico_extremos_iqr.csv", encoding="utf-8-sig")
print("Tabelas:", relatorios, "| Gráficos:", graficos)